<a href="https://colab.research.google.com/github/HeinZyrOo/CineMind/blob/main/CineMind_Phase_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install pandas numpy scikit-learn sentence-transformers faiss-cpu gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.2 MB/s eta 0:00:00


In [ ]:

import pandas as pd
import numpy as np
import json
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import faiss


In [ ]:
#movie dataset
#use TMDB 5000 Movie Dataset

from google.colab import files

uploaded = files.upload()

movies = pd.read_csv("tmdb_5000_movies.csv")

print("Dataset shape:", movies.shape)
display(movies.head())#Exploratory Data Analysis


Saving tmdb_5000_movies.csv to tmdb_5000_movies.csv
Dataset shape: (4803, 20)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [ ]:

#Data preprocessing
print(movies.columns.tolist())

print("\nMissing values:")
print(movies[["title", "overview", "genres", "keywords"]].isnull().sum())


['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

Missing values:
title       0
overview    3
genres      0
keywords    0
dtype: int64


In [ ]:
###Data preprocessing

#Extract genres and keywords
def extract_names(value):
    if pd.isna(value):
        return []

    try:
        data = json.loads(value)
        return [
            item["name"]
            for item in data
            if isinstance(item, dict) and "name" in item
        ]
    except (ValueError, TypeError):
        return []

#Select important columns
movies = movies[
    ["id", "title", "overview", "genres", "keywords"]
].copy()

#Remove missing values and duplicates
movies = movies.dropna(subset=["title", "overview"])
movies = movies.drop_duplicates(subset=["id"])

#Apply the extraction function
movies["genres_list"] = movies["genres"].apply(extract_names)
movies["keywords_list"] = movies["keywords"].apply(extract_names)


#Combine the movie features
movies["tags"] = (
    movies["overview"].fillna("") + " " +
    movies["genres_list"].apply(lambda x: " ".join(x)) + " " +
    movies["keywords_list"].apply(lambda x: " ".join(x))
)

movies["tags"] = (
    movies["tags"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

#Reset indices
movies = movies.reset_index(drop=True)

print("Processed movies:", len(movies))
display(movies[["title", "tags"]].head())


Processed movies: 4800


,title,tags
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


# **Note**

TF-IDF helps identify words that are important to a particular movie description relative to other descriptions.
TF-IDF mainly compares words and phrases.
TF-IDF assigns weights based on both how frequently a term occurs within a document and how common it is across documents.


Cosine similarity measures the angle between two vectors.
Closer to 1, more similar

In [ ]:
###Implement TF-IDF


tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",#Removes common English words
    ngram_range=(1, 2)#Uses both unigrams and bigrams.
)

tfidf_matrix = tfidf.fit_transform(movies["tags"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (4800, 10000)


In [ ]:

def recommend_tfidf(movie_title, top_k=5):
    matches = movies[
        movies["title"].str.lower() == movie_title.lower().strip() #Find the requested movie
    ]

    if matches.empty:
        return pd.DataFrame({"message": ["Movie not found"]})

    idx = matches.index[0]  #Get the movie index

####Calculate similarity

    similarities = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    similarities[idx] = -1  #Exclude the original movie

    top_indices = np.argsort(similarities)[::-1][:top_k]   #Sort by similarity

    return pd.DataFrame({
        "Movie": movies.iloc[top_indices]["title"].values,
        "Similarity": similarities[top_indices].round(4)
    })


display(recommend_tfidf("Interstellar"))


,Movie,Similarity
0,Silent Running,0.2879
1,Mission to Mars,0.2523
2,2001: A Space Odyssey,0.2496
3,Space Cowboys,0.2347
4,Armageddon,0.2276


# **Note**
Sentence Transformers generates embeddings (numerical vectors), and FAISS uses those embeddings to find movies with similar meanings.
      

In [ ]:
###Semantic recommendation


#Sentence Transformers is a framework for generating meaningful vector representations of sentences, paragraphs, and other text.

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"## pretrained transformer-based embedding model
)

print("Model loaded successfully")
print("Embedding dimension:", model.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully
Embedding dimension: 384


/tmp/ipykernel_1200/1837246239.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


# **Note**

embedding is a numerical vector representing information about an input.
convert all movie descriptions into embeddings.
compare them mathematically.

In [ ]:
###Generate embeddings

embeddings = model.encode(
    movies["tags"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Embedding shape: (4800, 384)


Process A — Preparing the movie embeddings


*   load all movie descriptions.
*   Sentence Transformer converts every description into an embedding.
*   store those embeddings in FAISS.
*   FAISS is now ready to find similar movies.





In [ ]:
###Build FAISS index(Facebook AI Similarity Search)

dimension = embeddings.shape[1]#Get embedding dimension

index = faiss.IndexFlatIP(dimension)#Create FAISS index
index.add(embeddings)

print("Movies indexed:", index.ntotal)


Movies indexed: 4800


# **Note**
Faiss =
a library for efficient nearest-neighbor search over vectors.
to find vectors that are mathematically close to a query vector.

In [ ]:
###Recommend movies using FAISS
def recommend_semantic(movie_title, top_k=5):
    matches = movies[
        movies["title"].str.lower() == movie_title.lower().strip()
    ]

    if matches.empty:
        return pd.DataFrame({"message": ["Movie not found"]})

    idx = matches.index[0] #Find the movie

    query_vector = embeddings[idx:idx + 1]#Get its embedding

    k = min(top_k + 1, index.ntotal)
    scores, indices = index.search(query_vector, k)#Search the index

    results = []

    for score, movie_idx in zip(scores[0], indices[0]):
        if movie_idx == idx: #Search the index
            continue

        results.append({ #Store the recommendations
            "Movie": movies.iloc[movie_idx]["title"],
            "Similarity": round(float(score), 4)
        })

        if len(results) == top_k:#Stop after five results
            break

    return pd.DataFrame(results)


display(recommend_semantic("Interstellar"))


,Movie,Similarity
0,Mission to Mars,0.5471
1,Prometheus,0.5346
2,Gattaca,0.5273
3,The Inhabited Island,0.5271
4,Star Trek Beyond,0.5259


In [ ]:
###Compare TF-IDF and Sentence Transformers

def compare_models(movie_title, top_k=5):
    tfidf_results = recommend_tfidf(movie_title, top_k)
    semantic_results = recommend_semantic(movie_title, top_k)

    print(f"Movie: {movie_title}")
    print("\n=== TF-IDF Recommendations ===")
    display(tfidf_results)

    print("\n=== Semantic Recommendations ===")
    display(semantic_results)


compare_models("Interstellar")
compare_models("The Dark Knight")
compare_models("Avatar")


Movie: Interstellar

=== TF-IDF Recommendations ===


,Movie,Similarity
0,Silent Running,0.2879
1,Mission to Mars,0.2523
2,2001: A Space Odyssey,0.2496
3,Space Cowboys,0.2347
4,Armageddon,0.2276



=== Semantic Recommendations ===


,Movie,Similarity
0,Mission to Mars,0.5471
1,Prometheus,0.5346
2,Gattaca,0.5273
3,The Inhabited Island,0.5271
4,Star Trek Beyond,0.5259


Movie: The Dark Knight

=== TF-IDF Recommendations ===


...

,Movie,Similarity
0,The Dark Knight Rises,0.7539
1,Batman,0.7354
2,Batman Begins,0.7281
3,Batman v Superman: Dawn of Justice,0.7214
4,Batman & Robin,0.6902


Movie: Avatar

=== TF-IDF Recommendations ===


,Movie,Similarity
0,Aliens,0.3119
1,Alien³,0.2883
2,Mission to Mars,0.2771
3,Silent Running,0.2658
4,Moonraker,0.2554



=== Semantic Recommendations ===


,Movie,Similarity
0,Serenity,0.5175
1,Aliens,0.5174
2,Gattaca,0.5095
3,Star Trek Beyond,0.5084
4,Independence Daysaster,0.5040


In [ ]:
###Natural-language movie search
def recommend_by_description(description, top_k=5):

    query_embedding = model.encode(#Encode the user's description
        [description],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, index.ntotal)

    scores, indices = index.search(query_embedding, k)#Search FAISS

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "Movie": movies.iloc[idx]["title"],
            "Similarity": round(float(score), 4),
            "Overview": movies.iloc[idx]["overview"]
        })

    return pd.DataFrame(results)


query = "A science fiction movie about space exploration"

display(recommend_by_description(query))


,Movie,Similarity,Overview
0,Star Trek Beyond,0.5742,The USS Enterprise crew explores the furthest ...
1,Wing Commander,0.5468,The Hollywood version of the popular video gam...
2,Red Planet,0.5460,Astronauts search for solutions to save a dyin...
3,Interstellar,0.5446,Interstellar chronicles the adventures of a gr...
4,Moonraker,0.5178,During the transportation of a Space Shuttle a...


Process B — User searches for a movie

*   Sentence Transformer converts the user's query into an embedding.
*   FAISS compares that query embedding with the stored movie embeddings.
*   FAISS returns the five nearest vectors.
*   use the returned indices to retrieve movie titles from Pandas.





In [ ]:
###Build the Gradio interface
import gradio as gr

def movie_recommender(query, method):

    if not query.strip():
        return pd.DataFrame({
            "Message": ["Please enter a movie title or description."]
        })

    if method == "TF-IDF":
        return recommend_tfidf(query)

    elif method == "Semantic (Movie Title)":
        return recommend_semantic(query)

    else:
        return recommend_by_description(query)


demo = gr.Interface(
    fn=movie_recommender,

    inputs=[
        gr.Textbox(
            label="Movie Title or Description",
            placeholder="Interstellar or a movie about space exploration"
        ),

        gr.Radio(
            choices=[
                "TF-IDF",
                "Semantic (Movie Title)",
                "Semantic (Description)"
            ],
            value="Semantic (Movie Title)",
            label="Recommendation Method"
        )
    ],

    outputs=gr.Dataframe(
        label="Recommended Movies"
    ),

    title="CineMatch AI",
    description=(
        "AI-powered movie recommendation system "
        "using TF-IDF, Sentence Transformers and FAISS."
    ),

    examples=[
        ["Interstellar", "Semantic (Movie Title)"],
        ["The Dark Knight", "TF-IDF"],
        ["A science fiction movie about space", "Semantic (Description)"]
    ]
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://22e05d14172de9d6c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#Save the models
import os
import joblib

os.makedirs("models", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Save processed movie metadata
movies.to_csv(
    "data/processed/movies_processed.csv",
    index=False
)

# Save embeddings
np.save(
    "models/movie_embeddings.npy",
    embeddings
)

# Save FAISS index
faiss.write_index(
    index,
    "models/movie_faiss.index"
)

# Save TF-IDF vectorizer and matrix
joblib.dump(
    tfidf,
    "models/tfidf_vectorizer.joblib"
)

joblib.dump(
    tfidf_matrix,
    "models/tfidf_matrix.joblib"
)

print("All artifacts saved successfully!")


All artifacts saved successfully!


In [ ]:

movies = pd.read_csv(
    "data/processed/movies_processed.csv"
)

embeddings = np.load(
    "models/movie_embeddings.npy"
)

index = faiss.read_index(
    "models/movie_faiss.index"
)

tfidf = joblib.load(
    "models/tfidf_vectorizer.joblib"
)

tfidf_matrix = joblib.load(
    "models/tfidf_matrix.joblib"
)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Models loaded successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Models loaded successfully!
